# Preprocessing

In [1]:
%load_ext autoreload
%autoreload 2
import polars as pl
import pandas as pd
from utils import pipeline_v2, funcs
from transformers import pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from collections import defaultdict

c:\Users\Lucas\anaconda3\envs\text-mining\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reading and Splitting the Training Data

To optimize data preprocessing in the pipeline, we load the raw data into a **Polars DataFrame**, leveraging its multi-threaded engine to handle large files faster and more efficiently than pandas. 

To manage local memory constraints, the dataset is split into two equal partitions (`sample_df1` and `sample_df2`) and processed sequentially.

In [2]:
df = pl.read_csv('data/train.csv')

In [15]:
sample_df1, sample_df2 = train_test_split(
    df,
    train_size=0.5,
    stratify=df['tag'],
    random_state=80085
)

### Lyrics Preprocessing Overview

Each partition is mapped through `main_pipeline` via Polars' `map_elements` to clean and standardize the `lyrics` column:

- **Text Cleaning & Normalization:** Strips song metadata tags (e.g., *[Chorus]*, *[Verse]*), URLs, brackets, and punctuation, while expanding contractions.
- **Filtering & Tokenization:** Tokenizes text into individual words, discards non-alphabetic tokens, and removes standard English stopwords as well as common musical filler vocalizations (e.g., *"oh"*, *"yeah"*, *"la"*).
- **Linguistic Standardization:** Converts diacritics to ASCII, applies lowercasing, and lemmatizes tokens across multiple parts of speech.
- **Output Schema:** With `tokenized_output=True`, each song's lyrics are transformed into a list of cleaned tokens (`pl.List(pl.String)`) stored under `cleaned_lyrics`.

Once both batches finish processing, they are combined back into a single unified dataset (`cleaned_df`) using vertical concatenation (`pl.concat`).

In [ ]:
from utils.pipeline_v2 import main_pipeline

sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True),
        return_dtype=pl.List(pl.String)
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True),
        return_dtype=pl.List(pl.String)
    )
    .alias("cleaned_lyrics")
)

cleaned_df = pl.concat([sample_df1, sample_df2])

## Checkpointing & Class Distribution Overview

To preserve our preprocessed data and avoid redundant recomputation, we export the DataFrame to **Parquet format** and reload it. Parquet provides efficient column-oriented compression and natively preserves complex nested schemas, such as columns containing lists of strings (`pl.List(pl.String)`).

In [ ]:
cleaned_df.select(['tag', 'cleaned_lyrics', 'artist', 'title']).write_parquet('data/data/cleaned_train_df_tokenized.parquet')

In [3]:
cleaned_df = pl.read_parquet(r'data\cleaned_train_df_tokenized.parquet')

In [5]:
cleaned_df.show(5)

tag,cleaned_lyrics,artist,title
str,list[str],str,str
"""rock""","[""another"", ""day"", … ""reappear""]","""Seduce (Metal)""","""The Mirror"""
"""rap""","[""lean"", ""chihiro"", … ""away""]","""Lean Chihiro""","""Shadows"""
"""pop""","[""earthquake"", ""goin"", … ""way""]","""Black Oak Arkansas""","""The Big Ones Still Coming"""
"""pop""","[""must"", ""understand"", … ""ohohoh""]","""Bucks Fizz""","""Whats Love Got to Do with it?"""
"""pop""","[""way"", ""look"", … ""eye""]","""The Ponys""","""Sad Eyes"""



### Class Balance Assessment
An initial inspection of the target genre distribution reveals a notable class imbalance, with **Rock**, **Pop**, and **Rap** making up the substantial majority of the dataset. Downstream modeling and evaluation strategies will need to account for this skew.

In [23]:
cleaned_df['tag'].value_counts()

tag,count
str,u32
"""rock""",25332
"""rb""",6203
"""pop""",55742
"""rap""",38584
"""country""",3466
"""misc""",5640


In [8]:
X = cleaned_df.select('cleaned_lyrics')
y = cleaned_df.select('tag')

## Class Balancing via Controlled Downsampling

To resolve the genre distribution skew observed during EDA, we apply `balance_classes` to the training set. Rather than collapsing every category strictly to the minority class count, the routine samples each genre into a bounded range around the target threshold to retain class diversity.

### Implementation Workflow
1. **Row Tracking & Temporary Merge:** To ensure feature matrix $X$ and labels $y$ remain perfectly synchronized during non-deterministic downsampling, we append a temporary row index (`__temp_id__`) and merge the labels as `target`.
2. **Resampling:** We invoke `balance_classes` on the unified DataFrame using a fixed random seed (`seed=80085`) for reproducibility.
3. **Partition Reconstruction:** The balanced records are split back into distinct `X_train_balanced` and `y_train_balanced` structures, dropping the temporary helper column.
4. **Saving the Dataset** For posterity the dataset is saved to Parquet.

In [10]:
from utils.funcs import balance_classes
# 1. Combine temporarily with a tracking index
# Convert y_train to a Series and rename it
y_series = y.to_series().alias("target")

combined = X.with_columns(
    pl.int_range(0, len(X)).alias("__temp_id__"),
    y_series,
)
# 2. Balance using the combined DataFrame
balanced_combined = balance_classes(combined, class_col="target", seed=80085)

# 3. Extract matching X_train and y_train
y_train_balanced = balanced_combined["target"]
X_train_balanced = balanced_combined.drop(["target", "__temp_id__"])

In [11]:
balanced_combined.write_parquet('data/balanced_train.parquet')
combined.write_parquet('data/combined_train.parquet')

## Dataset Branching: Co-Occurrence Matrix Generation

Rather than restricting downstream modeling to a single architecture, we branch our data preparation to support varied embedding strategies, ranging from traditional distributional methods (Word2Vec, GloVe) to deep sequence models (Transformers).

### Co-Occurrence Computation
Count-based embeddings require pairwise co-occurrence frequencies across the corpus. Computing a full co-occurrence matrix naively can trigger SciPy out-of-memory errors due to quadratic vocabulary growth. 

We run `cooccurrence_matrix_generator` to slice matrix operations into manageable chunks, prune self-co-occurrences, and persist the output as a compact Parquet file (`data/cooccurrence_matrix.parquet`).

In [4]:
from utils.funcs import cooccurrence_matrix_generator

df_cooccurrence = cooccurrence_matrix_generator(cleaned_df, "cleaned_lyrics")
df_cooccurrence.write_parquet("data/cooccurrence_matrix.parquet")

Computing pairs in blocks: 100%|██████████| 48/48 [00:52<00:00,  1.10s/it]


Assembling final co-occurrence table...
